In [2]:
!pip install -U transformers trl peft datasets accelerate -q

In [3]:
import torch
import gc
import os
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, PeftModel, get_peft_model
from trl import SFTConfig, SFTTrainer

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


### Utility functions (provided)

In [4]:
def clear_memory():
    """Free GPU memory between experiments."""
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        print(f"GPU allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
        print(f"GPU reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")


def print_trainable_params(model):
    """Display trainable vs total parameter counts."""
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Trainable parameters: {trainable:>12,}  ({100 * trainable / total:.4f}%)")
    print(f"Total parameters:     {total:>12,}")


def generate_response(model, tokenizer, prompt, max_new_tokens=256):
    """Generate a response from a model given a prompt string."""
    messages = [{"role": "user", "content": prompt}]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response

---
## Part 1 — Configuration

### Task 1: Define all hyperparameters

Set the following configuration variables:

| Variable | Value |
|---|---|
| `MODEL_NAME` | `"TinyLlama/TinyLlama-1.1B-Chat-v1.0"` |
| `DATASET_NAME` | `"databricks/databricks-dolly-15k"` |
| `NUM_TRAIN_SAMPLES` | `1000` |
| `MAX_SEQ_LENGTH` | `512` |
| `LORA_R` | `8` |
| `LORA_ALPHA` | `16` |
| `LORA_DROPOUT` | `0.05` |
| `LORA_TARGET_MODULES` | `["q_proj", "v_proj"]` (attention only) |
| `LEARNING_RATE` | `2e-4` |
| `NUM_EPOCHS` | `1` |
| `BATCH_SIZE` | `4` |
| `GRAD_ACCUM_STEPS` | `4` |
| `OUTPUT_DIR` | `"./sft-lora-tinyllama"` |

In [5]:
#from transformers.models.canine.tokenization_canine import MASK
# Part1 Configuration
# Model and Dataset configuration

# Pretrained casual language model used as the base model
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

#Instruction-following dataset used for Supervised Fine Tuning
DATASET_NAME = "databricks/databricks-dolly-15k"

#Number of Training samples selected from Dolly-15K
NUM_TRAIN_SAMPLES = 1000

#Maximum no of tokens allowed for each training sequence
MAX_SEQ_LENGTH = 512

# LoRA configuration

# Smaller rank for fewer trainable parameter and lower memory use
LORA_R = 8

# Effective LORA scaling is approximately LORA_ALPHA /LORA_R
LORA_ALPHA = 16

# Dropout applied to LORA adapters for regularization
LORA_DROPOUT = 0.05

# Apply LORA for query and value projection layers
LORA_TARGET_MODULES = ["q_proj", "v_proj"]


#Training Configurations

#Learning Rate
LEARNING_RATE = 2e-4

#Number of complete passes through the training dataset
NUM_EPOCHS =1

#Num of samples processed per device in one forward pass
BATCH_SIZE = 4

# Accumulate gradients across multiple mini-batches before performing one optimizer update.
GRAD_ACCUM_STEPS = 4

# Directory used for checkpoints and final adapter artifacts.
OUTPUT_DIR = "./sft-lora-tinyllama"


---
## Part 2 — Load Model and Tokenizer


In [6]:
# Load the tokenizer associated with the TinyLLama checkpoint

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

#TinyLlama may not define a separate padding token. For causal language models, using the EOS token as padding

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load the pre-trained TinyLlama model in FP16.
# FP16 reduces GPU memory usage compared with FP32.
# device_map="auto" places the model on the available GPU automatically.

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16, #means model parameters are loaded primarily in 16-bit floating point instead of FP32.
    device_map="auto",
)


#Calculate the total number of parameters in the base model
total_parameters = sum(parameter.numel() for parameter in model.parameters())
print(f"Total parameters: {total_parameters:,}")
print(f"Model device: {next(model.parameters()).device}")
print(f"Model dtype : {next(model.parameters()).dtype}")

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Total parameters: 1,100,048,384
Model device: cuda:0
Model dtype : torch.float16


---
## Part 3 — Test Base Model

### Task 3: Generate responses from the base model


In [8]:
# ============================================================
# PART 3: TEST BASE MODEL
# ============================================================
import textwrap
# Fixed evaluation prompts used to measure the behavious of the base model before LORA fine tuning
test_prompts =[    "Explain treaty reinsurance and facultative reinsurance in exactly 3 bullet points using simple business language.",
        "List exactly 3 key differences between proportional and non-proportional treaty reinsurance.",
        "Explain what an insurance policy exclusion is in 2-3 sentences for a non-technical business user.",
        "Summarize the role of a reinsurer in exactly 4 concise bullet points.",
        "Explain the difference between an insurer, reinsurer, and broker in exactly 3 bullet points."]
# Store the base responses to compare the LoRA fine tuned model later
base_responses = []
for prompt in test_prompts:
    response = generate_response(model = model,
                                 tokenizer = tokenizer,
                                 prompt = prompt)
    base_responses.append(response)
    print ("=" *50)
    print (f"Test Prompt to TinyLlama-1.1B Model:'\n{prompt}")
    print ("*" *50)
    print (f"Test Prompt Response:\n{textwrap.fill(response, width=250)}")
    print ("*" *50)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Test Prompt to TinyLlama-1.1B Model:'
Explain treaty reinsurance and facultative reinsurance in exactly 3 bullet points using simple business language.
**************************************************
Test Prompt Response:
1. Treaty reinsurance: This type of reinsurance is a binding contract between two reinsurance companies that guarantees coverage for a specified risk or event. Treaty reinsurance is commonly used in the insurance industry for complex risks that
cannot be adequately covered by traditional reinsurance.  2. Facultative reinsurance: Facultative reinsurance is a type of reinsurance that provides a policyholder with the option to purchase additional coverage for a specified risk. Facultative
reinsurance allows the policyholder to increase their coverage if needed, but it also increases their risk exposure.  3. Treaty reinsurance: Treaty reinsurance is a binding contract between two reinsurance companies that guarantees coverage for a
specified risk or event. This type of

[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Test Prompt to TinyLlama-1.1B Model:'
List exactly 3 key differences between proportional and non-proportional treaty reinsurance.
**************************************************
Test Prompt Response:
1. Premium structure: Non-proportional treaty reinsurance premiums are generally lower than proportional treaty reinsurance premiums. This is because non-proportional treaty reinsurance policies are not based on the percentage of loss covered by the
insurer, but instead on the actual loss incurred by the insured.  2. Loss allocation: Non-proportional treaty reinsurance policies usually have a loss allocation formula that divides the losses between the insurer and the insured based on the actual
loss incurred by the insured. Proportional treaty reinsurance policies have a fixed percentage of the loss that is allocated to the insurer and a fixed percentage that is allocated to the insured.  3. Coordination: Non-proportional treaty reinsurance
policies often involve a coordination mechani

[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Test Prompt to TinyLlama-1.1B Model:'
Explain what an insurance policy exclusion is in 2-3 sentences for a non-technical business user.
**************************************************
Test Prompt Response:
An insurance policy exclusion is a clause in a policy that specifies which covered events or circumstances are excluded from coverage. This can be done to exclude specific types of events or situations that would normally be covered by the policy,
such as a loss due to natural disasters or a specific type of injury. The exclusion can be added to the policy at the time of purchase or updated during the policy period. Insurance policies may also exclude coverage for certain types of claims, such
as those related to theft, fire, or vandalism.
**************************************************


[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Test Prompt to TinyLlama-1.1B Model:'
Summarize the role of a reinsurer in exactly 4 concise bullet points.
**************************************************
Test Prompt Response:
1. Protects the client from financial losses due to unforeseen events such as natural disasters, terrorism, or pandemics. 2. Reimburses the client for losses incurred due to unforeseen events. 3. Negotiates insurance policies with insurance companies
to provide the best possible coverage at the most reasonable premium. 4. Ensures that the insured receives the maximum benefit possible under the policy terms.
**************************************************
Test Prompt to TinyLlama-1.1B Model:'
Explain the difference between an insurer, reinsurer, and broker in exactly 3 bullet points.
**************************************************
Test Prompt Response:
Insurance companies, reinsurers, and brokers are all types of companies that provide insurance services to customers. Here are the 3 bullet points to exp

---
## Part 4 — Prepare the Dataset

### Task 4: Load and format the Dolly dataset


In [9]:
# ============================================================
# PART 4: PREPARE THE DOLLY DATASET
# ============================================================
# Load the dataset

raw_dataset = load_dataset(DATASET_NAME, split="train")

# ------------------------------------------------------------
# Convert Dolly records into chat format
# ------------------------------------------------------------

def format_dolly_to_chat(example):

  """
    Convert a Dolly instruction-response record into the chat structure expected for supervised fine-tuning.

    If context is available, it is appended to the instruction using a newline separator.

    Parameters
    ----------
    example : dict
        A single Dolly dataset record containing: instruction, context, response, and category.

    Returns
    -------
    dict
        Dictionary containing a 'messages' field with user and assistant messages.
    """

  instruction = example["instruction"].strip()
  context = example["context"].strip()
  response = example["response"].strip()





  if len(context) > 0:
    user_content = f"{instruction}\n\n{context}"
  else:
    user_content = instruction

  return {"messages": [
      {"role": "user", "content": user_content},
      {"role": "assistant", "content": response}
  ]}


  # ------------------------------------------------------------
  # Step 3: Shuffle, sample, and format the dataset
  # ------------------------------------------------------------

sample_dataset = (raw_dataset.shuffle(seed=42).select(range(NUM_TRAIN_SAMPLES)))

formatted_dataset = sample_dataset.map(format_dolly_to_chat, remove_columns = sample_dataset.column_names)

# ------------------------------------------------------------

assert len(formatted_dataset) == NUM_TRAIN_SAMPLES, \
    "Formatted dataset size does not match NUM_TRAIN_SAMPLES."

assert "messages" in formatted_dataset.column_names, \
    "Expected 'messages' column was not created."

assert len(formatted_dataset[0]["messages"]) == 2, \
    "Each training example must contain one user and one assistant message."

print(f"Formatted dataset size: {len(formatted_dataset)}")
print("\nExample formatted entry:")
print(formatted_dataset[0])
print("\nDataset validation passed.")


README.md:   0%|          | 0.00/8.20k [00:00<?, ?B/s]

databricks-dolly-15k.jsonl: reconstructing file:   0%|          |  0.00B / 13.1MB            

databricks-dolly-15k.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Formatted dataset size: 1000

Example formatted entry:
{'messages': [{'content': 'Who were the children of the legendary Garth Greenhand, the High King of the First Men in the series A Song of Ice and Fire?', 'role': 'user'}, {'content': 'Garth the Gardener, John the Oak, Gilbert of the Vines, Brandon of the Bloody Blade, Foss the Archer, Owen Oakenshield, Harlon the Hunter, Herndon of the Horn, Bors the Breaker, Florys the Fox, Maris the Maid, Rose of the Red Lake, Ellyn Ever Sweet, Rowan Gold-Tree', 'role': 'assistant'}]}

Dataset validation passed.


---
## Part 5 — Configure LoRA

### Task 5: Create the LoRA config and inspect trainable parameters

1. Create a `LoraConfig` using the hyperparameters from Task 1. Set `bias="none"` and `task_type="CAUSAL_LM"`.
2. Apply it to the model with `get_peft_model`.
3. Use `print_trainable_params` to display the trainable vs total parameters.
4. Print the LoRA-wrapped structure of one attention layer (e.g., `q_proj` of layer 0).

In [11]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [12]:
#LORA Configuration

# ------------------------------------------------------------
# Step 1: Define the LoRA configuration
# ------------------------------------------------------------


lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)

# Insert the LORA Adapters into the Base Model

peft_model = get_peft_model(model, lora_config)

# ------------------------------------------------------------
# Step 3: Display the trainable parameters
# ------------------------------------------------------------

print_trainable_params(peft_model)


# ------------------------------------------------------------
# Step 4: Inspect one LoRA-wrapped attention projection layer
# ------------------------------------------------------------

print("\nLoRA-wrapped q_proj from layer 0:")

print(peft_model.base_model.model.model.layers[0].self_attn.q_proj)


Trainable parameters:    1,126,400  (0.1023%)
Total parameters:     1,101,174,784

LoRA-wrapped q_proj from layer 0:
lora.Linear(
  (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
  (lora_dropout): ModuleDict(
    (default): Dropout(p=0.05, inplace=False)
  )
  (lora_A): ModuleDict(
    (default): Linear(in_features=2048, out_features=8, bias=False)
  )
  (lora_B): ModuleDict(
    (default): Linear(in_features=8, out_features=2048, bias=False)
  )
  (lora_embedding_A): ParameterDict()
  (lora_embedding_B): ParameterDict()
  (lora_magnitude_vector): ModuleDict()
)


---
## Part 6 — Train

### Task 6: Set up SFTTrainer and run training

1. Delete the PEFT-wrapped model and call `clear_memory()`. Reload the base model fresh (SFTTrainer applies LoRA itself via `peft_config`).
2. Create an `SFTConfig` with the hyperparameters from Task 1. Include:
   - Cosine LR scheduler, warmup ratio 0.03, weight decay 0.01
   - `gradient_checkpointing=True`
   - bf16 if supported, else fp16
   - `max_length=MAX_SEQ_LENGTH`
   - Logging every 10 steps
3. Create an `SFTTrainer` with the model, training args, dataset, tokenizer, and `peft_config`.
4. Call `trainer.train()` and print the final loss.

In [13]:
# Part 6 - Train the LoRA Adapter
# ------------------------------------------------------------
# Step 1: Remove the inspection PEFT model and clear GPU memory
# ------------------------------------------------------------
if "peft_model" in globals():
    del peft_model

if "model" in globals():
    del model

clear_memory()

#use BF16 when the GPU supports it; otherwise fall back to FP16

bf16_supported = torch.cuda.is_bf16_supported()
fp16_supported = not bf16_supported

training_dtype = (
    torch.bfloat16
    if bf16_supported
    else torch.float16
)

print(f"BF16 supported: {bf16_supported}")
print(f"Training precision: {'BF16' if bf16_supported else 'FP16'}")

# ------------------------------------------------------------
# Step 3: Reload a fresh TinyLlama base model
# ------------------------------------------------------------

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)

# Gradient checkpoint is used during Training, so disable KV-Cache because cache is incompatible with this set-up

model.config.use_cache = False


# ------------------------------------------------------------
# Step 4: Configure Supervised Fine Tuning
# ------------------------------------------------------------

sft_config = SFTConfig(
    output_dir = OUTPUT_DIR,
    learning_rate = LEARNING_RATE,
    num_train_epochs=  NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,

    #Learning Rate Scheduling and regularization

    lr_scheduler_type ="cosine",
    warmup_steps = 0.03,
    weight_decay=0.01,

    #Memory Optimization
    bf16 = bf16_supported,
    fp16 = fp16_supported,

    #Maximum tokenized sequence length
    max_length = MAX_SEQ_LENGTH,


    # Training progress reporting

    logging_steps = 10,
    logging_strategy ="steps",

    # Reproducibility
    seed = 42,

    # Avoid unnecessary external experiment trackers
    report_to="none"


)

# ------------------------------------------------------------
# Step 5: Create the SFTTrainer
# ------------------------------------------------------------

trainer = SFTTrainer(
    model = model,
    train_dataset = formatted_dataset,
    # SFTTrainer injects the LoRA adapters into the clean base model using this configuration.
    peft_config = lora_config,
    # Current TRL uses processing_class instead of the older tokenizer argument.
    processing_class = tokenizer,
    args = sft_config,)



# ------------------------------------------------------------
# Step 6: Verify trainable parameters before training
# ------------------------------------------------------------

print_trainable_params(trainer.model)

# ------------------------------------------------------------
# Step 7: Run training
# ------------------------------------------------------------

train_result = trainer.train()


# ------------------------------------------------------------
# Step 8: Report the final training loss
# ------------------------------------------------------------

final_loss = train_result.training_loss

print("*" *50)
print("Training complete.")
print("*" *50)
print(f"Final training loss: {final_loss}")
print("*" *50)

GPU allocated: 2.06 GB
GPU reserved:  2.10 GB
BF16 supported: True
Training precision: BF16


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (4118 > 2048). Running this sequence through the model will result in indexing errors


Building labels for train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

...

Dropping fully masked examples from train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Trainable parameters:    1,126,400  (0.1023%)
Total parameters:     1,101,174,784


Step,Training Loss
10,2.114207
20,1.881418
30,1.755974
40,1.732297


Step,Training Loss
10,2.114207
20,1.881418
30,1.755974
40,1.732297
50,1.779100
60,1.720276


**************************************************
Training complete.
**************************************************
Final training loss: 1.8297754545060416
**************************************************


---
## Part 7 — Save the Adapter

### Task 7: Save the LoRA adapter and print its size

1. Save the trained model to `f"{OUTPUT_DIR}/final-adapter"`.
2. Also save the tokenizer to the same path.
3. Compute and print the adapter size in MB.

In [14]:
# ============================================================
# PART 7: SAVE THE LoRA ADAPTER
# ============================================================

import os

# ------------------------------------------------------------
# Step 1: Define the final adapter directory
# ------------------------------------------------------------

final_adapter_path = f"{OUTPUT_DIR}/final-adapter"

# ------------------------------------------------------------
# Step 2: Save the trained model and tokenizer
# ------------------------------------------------------------

trainer.model.save_pretrained(final_adapter_path)

# ------------------------------------------------------------
# Step 3: Save the tokenizer
# ------------------------------------------------------------

tokenizer.save_pretrained(final_adapter_path)

# ------------------------------------------------------------
# Step 4: Compute and print the adapter size
# ------------------------------------------------------------

adapter_size = sum(
    os.path.getsize(os.path.join(final_adapter_path, file))
    for file in os.listdir(final_adapter_path)
    if os.path.isfile(os.path.join(final_adapter_path, file))
)

adapter_size_mb = adapter_size / (1024 **2)

print(f"Adapter size: {adapter_size_mb} MB")

# ------------------------------------------------------------
# Step 5: Display adapter information
# ------------------------------------------------------------

print(f"Adapter saved to : {final_adapter_path}")
print(f"Adapter size     : {adapter_size_mb:.2f} MB")


Adapter size: 7.765937805175781 MB
Adapter saved to : ./sft-lora-tinyllama/final-adapter
Adapter size     : 7.77 MB


---
## Part 8 — Evaluate

### Task 8: Compare base vs fine-tuned responses

1. Use `trainer.model` (already has LoRA weights) to generate responses for the same `test_prompts`.
2. Print a side-by-side comparison of base vs fine-tuned responses.

In [17]:
# ============================================================
# PART 8: EVALUATE BASE VS FINE-TUNED MODEL
# ============================================================

# ------------------------------------------------------------
# Step 1: Prepare the fine-tuned model for inference
# ------------------------------------------------------------

fine_tuned_LORA_model = trainer.model

# Switch from training mode to evaluation mode.

fine_tuned_LORA_model.eval()


# ------------------------------------------------------------
# Step 2: Generate responses using the same test prompts
# ------------------------------------------------------------

fine_tuned_responses = []

for prompt in test_prompts:

    response = generate_response(model = fine_tuned_LORA_model,
                                 tokenizer = tokenizer,
                                 prompt = prompt)

    fine_tuned_responses.append(response)



# ------------------------------------------------------------
# Step 3: Compare base and fine-tuned responses
# ------------------------------------------------------------

for index, prompt in enumerate(test_prompts):

    print ("=" *50)
    print (f"Test Prompt:'{index+1}':\n{prompt}")
    print ("*" *50)

    print (f"Base Model Response:\n{textwrap.fill(base_responses[index], width=250)}")
    print ("*" *50)

    print (f"Fine-Tuned Model Response:\n{textwrap.fill(fine_tuned_responses[index],width=200)}")
    print ("*" *50)

    #{textwrap.fill(response, width=250)}

    print()





[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

Test Prompt:'1':
Explain treaty reinsurance and facultative reinsurance in exactly 3 bullet points using simple business language.
**************************************************
Base Model Response:
1. Treaty reinsurance: This type of reinsurance is a binding contract between two reinsurance companies that guarantees coverage for a specified risk or event. Treaty reinsurance is commonly used in the insurance industry for complex risks that
cannot be adequately covered by traditional reinsurance.  2. Facultative reinsurance: Facultative reinsurance is a type of reinsurance that provides a policyholder with the option to purchase additional coverage for a specified risk. Facultative
reinsurance allows the policyholder to increase their coverage if needed, but it also increases their risk exposure.  3. Treaty reinsurance: Treaty reinsurance is a binding contract between two reinsurance companies that guarantees coverage for a
specified risk or event. This type of reinsurance is common

---
## Done

You have successfully fine-tuned TinyLlama with LoRA on the Dolly dataset. Review your base vs fine-tuned outputs and note the differences in instruction-following quality.